# Notebook 1: Data Extraction
## India MSME Credit Stress Index Project
**Author:** Zaara Akbar Shaikh  
**Date:** May 2026  
**Purpose:** Extract and clean all three data pillars from primary sources

### Data Sources:
- Pillar 1: RBI Table 45 (MSME Credit Growth) — Excel
- Pillar 2: RBI Table 53 (Banking NPA) — Excel  
- Pillar 3: CGTMSE Guarantees (SIDBI PDF) — pdfplumber extraction

In [1]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import pandas as pd
import numpy as np
import pdfplumber
import os

print("All libraries loaded")

All libraries loaded


In [2]:
# ============================================================
# CELL 2 — Extract RBI Table 45 (MSME Credit Data)
# ============================================================
# Business question: How fast has MSME credit grown year on year?

msme_credit_data = {
    'fiscal_year': ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
    'micro_small_priority_cr': [1206003, 1428326, 1643084, 1974191, 2240503],
    'medium_priority_cr': [258462, 374492, 423566, 490703, 601451]
}

df_msme = pd.DataFrame(msme_credit_data)

# Combined MSME credit (Micro + Small + Medium)
df_msme['total_msme_priority_cr'] = (
    df_msme['micro_small_priority_cr'] + df_msme['medium_priority_cr']
)

# Year-on-year growth rate — this becomes Pillar 1
df_msme['msme_credit_growth_pct'] = (
    df_msme['total_msme_priority_cr'].pct_change() * 100
).round(2)

df_msme['source'] = 'RBI Handbook Table 45'

print("MSME Credit Data:")
print(df_msme.to_string())

df_msme.to_csv('../data/processed/rbi_msme_credit.csv', index=False)
print("\nSaved: rbi_msme_credit.csv")


MSME Credit Data:
  fiscal_year  micro_small_priority_cr  medium_priority_cr  total_msme_priority_cr  msme_credit_growth_pct                 source
0     2020-21                  1206003              258462                 1464465                     NaN  RBI Handbook Table 45
1     2021-22                  1428326              374492                 1802818                   23.10  RBI Handbook Table 45
2     2022-23                  1643084              423566                 2066650                   14.63  RBI Handbook Table 45
3     2023-24                  1974191              490703                 2464894                   19.27  RBI Handbook Table 45
4     2024-25                  2240503              601451                 2841954                   15.30  RBI Handbook Table 45

Saved: rbi_msme_credit.csv


In [3]:
# ============================================================
# CELL 3 — Extract RBI Table 53 (NPA Data)
# ============================================================
# Business question: How has banking system stress evolved over 12 years?

npa_data = {
    'fiscal_year': ['2012-13','2013-14','2014-15','2015-16','2016-17',
                    '2017-18','2018-19','2019-20','2020-21','2021-22',
                    '2022-23','2023-24'],
    'gross_npa_pct': [3.2, 3.8, 4.3, 7.5, 9.3,
                      11.2, 9.1, 8.2, 7.3, 5.8,
                      3.9, 2.7]
}

df_npa = pd.DataFrame(npa_data)
df_npa['npa_yoy_change'] = df_npa['gross_npa_pct'].diff().round(2)
df_npa['stress_signal'] = df_npa['npa_yoy_change'].apply(
    lambda x: 'Sharp Deterioration' if x > 1
    else 'Mild Deterioration' if x > 0
    else 'Sharp Improvement' if x < -1
    else 'Mild Improvement' if pd.notna(x)
    else 'Baseline'
)
df_npa['source'] = 'RBI Handbook Table 53'

print("\nNPA Data:")
print(df_npa.to_string())

df_npa.to_csv('../data/processed/rbi_npa.csv', index=False)
print("\nSaved: rbi_npa.csv")


NPA Data:
   fiscal_year  gross_npa_pct  npa_yoy_change        stress_signal                 source
0      2012-13            3.2             NaN             Baseline  RBI Handbook Table 53
1      2013-14            3.8             0.6   Mild Deterioration  RBI Handbook Table 53
2      2014-15            4.3             0.5   Mild Deterioration  RBI Handbook Table 53
3      2015-16            7.5             3.2  Sharp Deterioration  RBI Handbook Table 53
4      2016-17            9.3             1.8  Sharp Deterioration  RBI Handbook Table 53
5      2017-18           11.2             1.9  Sharp Deterioration  RBI Handbook Table 53
6      2018-19            9.1            -2.1    Sharp Improvement  RBI Handbook Table 53
7      2019-20            8.2            -0.9     Mild Improvement  RBI Handbook Table 53
8      2020-21            7.3            -0.9     Mild Improvement  RBI Handbook Table 53
9      2021-22            5.8            -1.5    Sharp Improvement  RBI Handbook Table 53

In [4]:
# ============================================================
# CELL 4 — Extract CGTMSE Data (Third Pillar)
# ============================================================
# Business question: How aggressively is government backing MSME credit?

pdf_path = r"C:\Users\Zaara\Desktop\MSME\data\raw\sidbi\sidbi_pulse_2025_05.pdf"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[33]
    tables = page.extract_tables()
    raw_table = tables[0]

df_cgtmse_raw = pd.DataFrame(raw_table)
df_cgtmse_raw.columns = df_cgtmse_raw.iloc[1]
df_cgtmse_raw = df_cgtmse_raw.iloc[2:].reset_index(drop=True)

# Remove cumulation row
df_cgtmse = df_cgtmse_raw[
    ~df_cgtmse_raw['Period'].str.contains('Cumulation', na=False)
].copy()

# Clean Period column
df_cgtmse['Period'] = df_cgtmse['Period'].str.replace(
    r'\s*Till.*$', '', regex=True
).str.strip()

# Clean numeric columns
df_cgtmse['No.'] = df_cgtmse['No.'].str.replace(',', '').astype(int)
df_cgtmse['Amt. (In Crs)'] = (
    df_cgtmse['Amt. (In Crs)'].str.replace(',', '').astype(int)
)

df_cgtmse.columns = ['fiscal_year', 'guarantees_count', 'guarantee_amt_cr']
df_cgtmse['guarantee_growth_pct'] = (
    df_cgtmse['guarantee_amt_cr'].pct_change() * 100
).round(2)
df_cgtmse['source'] = 'SIDBI MSME Pulse May 2025, Page 29'

print("\nCGTMSE Data:")
print(df_cgtmse.to_string())

df_cgtmse.to_csv('../data/processed/cgtmse_guarantees.csv', index=False)
print("\nSaved: cgtmse_guarantees.csv")


CGTMSE Data:
  fiscal_year  guarantees_count  guarantee_amt_cr  guarantee_growth_pct                              source
0  FY 2019-20            846650             45851                   NaN  SIDBI MSME Pulse May 2025, Page 29
1  FY 2020-21            835592             36899                -19.52  SIDBI MSME Pulse May 2025, Page 29
2  FY 2021-22            717020             56172                 52.23  SIDBI MSME Pulse May 2025, Page 29
3  FY 2022-23           1165786            104781                 86.54  SIDBI MSME Pulse May 2025, Page 29
4  FY 2023-24           1724073            202807                 93.55  SIDBI MSME Pulse May 2025, Page 29
5  FY 2024-25           2715275            305507                 50.64  SIDBI MSME Pulse May 2025, Page 29

Saved: cgtmse_guarantees.csv


### Q) Why did CGTMSE guarantees fall 19.5% in FY2021, then grow 52%, 87%, 94% in the next three years? What was ECLGS and what did it signal about MSME credit risk?

In [5]:
# ── Cell 5: Load cleaned data to MySQL ────────────────────────────
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load credentials from .env — never hardcoded
load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

# Load to MySQL — append to existing tables
df_msme.to_sql('rbi_msme_credit',    engine, if_exists='append', index=False)
df_npa.to_sql('rbi_npa',             engine, if_exists='append', index=False)
df_cgtmse.to_sql('cgtmse_guarantees', engine, if_exists='append', index=False)

# Verify row counts
with engine.connect() as conn:
    for table in ['rbi_msme_credit', 'rbi_npa', 'cgtmse_guarantees']:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"✓ {table}: {count} rows")

✓ rbi_msme_credit: 5 rows
✓ rbi_npa: 12 rows
✓ cgtmse_guarantees: 6 rows
